In [0]:
%sql
-- Create a sample dataset as a temp view
CREATE OR REPLACE TEMP VIEW sample_metrics AS
SELECT 1 AS id, 'customer_metric' AS metric_name, 100 AS metric_value UNION ALL
SELECT Null, 'order_metric', 200 UNION ALL
SELECT 3, 'product_metric', NULL UNION ALL
SELECT 4, 'payment_metric', 400;

select count(distinct *) from sample_metrics;

In [0]:
tables = ['dqx_sandbox.dqx_bronze.input_table', 'dqx_sandbox.dqx_bronze.metrics_table', 'dqx_sandbox.dqx_bronze.output_table']
for table_name in tables:
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")

In [0]:
# dbutils.library.restartPython()

dbutils.widgets.removeAll()
dbutils.widgets.text("catalog_name", "", "catalog_name")
dbutils.widgets.text("config_schema_name", "", "config_schema_name")
dbutils.widgets.text("source_schema_name", "", "source_schema_name")

catalog_name = dbutils.widgets.get("catalog_name")
config_schema_name = dbutils.widgets.get("config_schema_name")
source_schema_name = dbutils.widgets.get("source_schema_name")


spark.sql(f"""
    INSERT OVERWRITE {catalog_name}.{config_schema_name}.dqx_rule_definitions 
    (rule_id, rule_name, rule_function, description, rule_type, rule_dimension, created_date, updated_date, argument_placeholder, is_arg_mendatory)
    VALUES 
    -- Null & Empty Checks
    ('R001', 'Not Null Check', 'is_not_null', 'Ensures the column contains no null values.', 'row_level', 'Completeness', current_timestamp(), current_timestamp(), '{{"column":"<col_name>"}}', false),
    ('R002', 'Not Empty Check', 'is_not_empty', 'Ensures string columns are not empty strings.', 'column_level', 'Completeness', current_timestamp(), current_timestamp(), '{{"column":"<col_name>"}}', false),
    
    -- Uniqueness & Identity
    ('R003', 'Unique Check', 'is_unique', 'Ensures all values in the column are distinct.', 'column_level', 'Uniqueness', current_timestamp(), current_timestamp(), '{{"columns":["<col_name>"]}}', false),
    ('R004', 'Primary Key Check', 'is_primary_key', 'Combines Not Null and Unique checks.', 'column_level', 'Uniqueness / Completeness', current_timestamp(), current_timestamp(), '{{"columns": ["pk_col1", "pk_col2"]}}', true),
    
    -- Numeric & Range Checks
    ('R005', 'In Range Check', 'is_not_in_range', 'Checks whether the values in the input column are within the provided limits (inclusive of both boundaries).', 'row_level', 'Validity', current_timestamp(), current_timestamp(), '{{"column":"<col_name>","min_limit":<min>,"max_limit":<max>}}', true),

    -- String & Pattern Checks
    -- Note: Regex usually needs quadruple braces if it contains its own curly brackets
    ('R006', 'Pattern Match', 'regex_match', 'Validates column values against a specified regex pattern.', 'row_level', 'Validity', current_timestamp(), current_timestamp(), '{{"pattern": "<regex_exp>', true),
    
    -- Foreign Key Check
    ('R007', 'Foreign Key Check', 'foreign_key', 'Ensures values exist in parent table.', 'column_level', 'Consistency / Integrity', current_timestamp(), current_timestamp(), '{{"columns":["<col_name>"],"ref_table": "catalog.schema.tablename", "ref_columns": ["<col_name>"]}}', true),
    
    -- SQL Expression Check
    ('R008', 'SQL Expression Check', 'sql_expression', 'Custom SQL validation.', 'column_level', 'Validity', current_timestamp(), current_timestamp(), '{{"expression": "column_a > column_b", "msg":"<message description>"}}', true),

    -- New checks
    ('R009', 'Min Max Check', 'min_max', 'Real min/max values were used.', 'column_level', 'Validity', current_timestamp(), current_timestamp(), '{{"column":"<col_name>","min":<value>,"max":<value>}}', true),
    ('R010', 'Not Less Than Check', 'is_not_less_than', 'Checks whether the values in the input column are not less than the provided minimum.', 'row_level', 'Validity', current_timestamp(), current_timestamp(), '{{"column":"<col_name>","limit":<min>}}', true)
""")

spark.sql(f"SELECT * FROM {catalog_name}.{config_schema_name}.dqx_rule_definitions").display()

In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from databricks.sdk import WorkspaceClient

from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule
from databricks.labs.dqx import check_funcs

In [0]:
# import databricks.labs.dqx
# print(help(databricks.labs.dqx))

In [0]:
df = table('dqx_sandbox.dqx_demo.customer')

In [0]:
rules = [
    DQRowRule(
        name="customer_id_not_null",
        column="customer_id",
        check_func=check_funcs.is_not_null,
        criticality="error" 
    )
]

# 4. Initialize Engine and Run
ws = WorkspaceClient()
engine = DQEngine(ws)

# This returns: (Passed Rows, Failed/Quarantined Rows)
valid_and_quarantine_df = engine.apply_checks(df, rules)

# 5. Review Results
print("--- Quarantine Report ---")
display(valid_and_quarantine_df)

In [0]:
%sql
desc detail dqx_sandbox.dqx_demo.customer 

In [0]:
%sql
select * from dqx_sandbox.dqx_bronze.customer

In [0]:
import yaml

In [0]:
%sql
WITH ranked_rules AS (
  SELECT m.column_name AS column, r.rule_dimension, r.rule_name, r.description as rule_description,
          m.criticality, m.arguments, m.is_active, m.rule_id, r.is_arg_mendatory,
          ROW_NUMBER() OVER (PARTITION BY m.table_name, m.column_name, r.rule_function ORDER BY m.updated_at DESC) as row_num
  FROM dqx_sandbox.dqx_config.dqx_rule_mappings m
  JOIN dqx_sandbox.dqx_config.dqx_rule_definitions r ON m.rule_id = r.rule_id
  WHERE m.table_name = 'dqx_sandbox.dqx_bronze.customer' AND m.is_active = true
) 
SELECT * FROM ranked_rules WHERE row_num = 1

In [0]:
%sql
with cte_a as (
  select 1 as batchId ,
	from_json('[{ "employeeId":1234,"sales" : 10000 },{ "employeeId":3232,"sales" : 30000 }]',
         'ARRAY<STRUCT<employeeId: BIGINT, sales: INT>>') as performance,
  current_timestamp() as insertDate
union all 
select 2 as batchId ,
  from_json('[{ "employeeId":1235,"sales" : 10500 },{ "employeeId":3233,"sales" : 32000 }]',
                'ARRAY<STRUCT<employeeId: BIGINT, sales: INT>>') as performance,
                current_timestamp() as insertDate
)
, cte_b as (
  select flatten(performance) from cte_a
)
select * from cte_b
